# Chapter 09 - Dataclasses

#### 1. Mechanical Refresher
`@dataclass` is a decorator that reads class annotations and generates common methods such as `__init__`, `__repr__`, and `__eq__`. It is useful for small structured records where the main job of the class is to store named fields, optional defaults, and a few simple methods.

#### 2. Minimal Working Example

In [ ]:
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

point = Point(2.0, 3.0)
same = Point(2.0, 3.0)
assert point == same
print(point)

The decorator runs after the class body is created. It sees `x` and `y` annotations and adds an initializer, readable representation, and equality behavior based on those fields.

#### 3. Modify Drills

**Modify Drill 1.** Add a default field and predict construction with and without the argument.

In [ ]:
from dataclasses import dataclass

@dataclass
class RunConfig:
    epochs: int
    lr: float = 0.1

assert RunConfig(3).lr == 0.1
assert RunConfig(3, 0.01).lr == 0.01
print("passed")

**Modify Drill 2.** Add a method that uses generated fields.

In [ ]:
from dataclasses import dataclass

@dataclass
class Range:
    start: int
    stop: int

    def width(self) -> int:
        return self.stop - self.start

assert Range(2, 7).width() == 5
print("passed")

**Modify Drill 3.** Use `field(default_factory=...)` for per-instance lists.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class History:
    values: list[float] = field(default_factory=list)

first = History()
second = History()
first.values.append(1.0)
assert first.values == [1.0]
assert second.values == []
print("passed")

#### 4. Break-and-Fix Drills

**Break-and-Fix Drill 1.** Break it by deleting the annotation on `name`. Predict why dataclass ignores the field, then restore the annotation.

In [ ]:
from dataclasses import dataclass

@dataclass
class Metric:
    name: str
    value: float

actual = Metric("loss", 0.5)
assert actual.name == "loss"
assert actual.value == 0.5
print("passed")

**Break-and-Fix Drill 2.** Break it by putting a non-default field after a default field. Predict the class-creation error, then put required fields first.

In [ ]:
from dataclasses import dataclass

@dataclass
class BatchSpec:
    size: int
    shuffle: bool = True

spec = BatchSpec(32)
assert spec.size == 32
assert spec.shuffle is True
print("passed")

#### 5. Self-Verification
Use asserts for generated behavior: construction, equality, representation when needed, and separate mutable defaults. A dataclass exercise is correct when the generated methods behave like the hand-written version you would otherwise need.

#### 6. Standalone Exercises

**Exercise 1.** Convert `User` into a dataclass. Expected behavior: equality is field-based.

In [ ]:
class User:
    name: str
    active: bool

first = None  # TODO: after adding @dataclass, create User("Ada", True).
second = None  # TODO: after adding @dataclass, create another equal User.
assert first == second and first is not second
print("passed")

**Exercise 2.** Add a default value. Expected behavior: `BatchOptions(16).shuffle is True`.

In [ ]:
from dataclasses import dataclass

@dataclass
class BatchOptions:
    size: int
    # TODO: add shuffle defaulting to True.

options = BatchOptions(16)
actual = getattr(options, "shuffle", None)
assert actual is True
print("passed")

**Exercise 3.** Add `default_factory` for a list. Expected behavior: histories do not share values.

In [ ]:
from dataclasses import MISSING, dataclass

@dataclass
class TrialHistory:
    values: list[float]

first = TrialHistory([])
second = TrialHistory([])
first.values.append(0.1)
assert first.values == [0.1]
assert second.values == []
assert TrialHistory.__dataclass_fields__["values"].default_factory is not MISSING
print("passed")

**Exercise 4.** Add a computed method. Expected behavior: `0.25`.

In [ ]:
from dataclasses import dataclass

@dataclass
class ErrorPair:
    prediction: float
    target: float

    def absolute_error(self) -> float:
        return 0.0  # TODO

assert ErrorPair(2.0, 2.25).absolute_error() == 0.25
print("passed")

**Exercise 5.** Use `frozen=True`. Expected behavior: field equality still works.

In [ ]:
from dataclasses import dataclass

@dataclass
class Token:
    text: str
    index: int

left = Token("hi", 0)
right = Token("hi", 0)
assert left == right
assert Token.__dataclass_params__.frozen is True
print("passed")

#### 7. Applied AI/ML Drill
**ML to Python mirror:** an experiment configuration is just a typed record with generated initialization and equality. **Python to ML mirror:** ML projects often keep training hyperparameters, metric summaries, and batch descriptions in dataclass-like records so functions can accept one clear object instead of many loose arguments.

**Applied Drill.** Complete a dataclass training config with defaults. Expected behavior: `steps_per_epoch()` returns `4`.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrainingConfig:
    examples: int
    batch_size: int
    lr: float = 0.1

    def steps_per_epoch(self) -> int:
        return 0  # TODO: ceiling-free integer division for exact case.

config = TrainingConfig(examples=20, batch_size=5)
assert config.steps_per_epoch() == 4
assert config.lr == 0.1
print("passed")

#### 8. Common Bugs
- Forgetting field annotations: dataclass only generates fields for annotated names.
- Putting required fields after default fields: the symptom is a class-creation `TypeError`.
- Using mutable defaults directly: use `field(default_factory=list)` for a fresh list per instance.
- Using dataclasses for behavior-heavy objects: the symptom is a record class accumulating too many unrelated methods.

#### 9. Compounding Drill

Combine dataclasses with Chapter 4 `@property`: store raw values and expose a computed metric.

In [ ]:
from dataclasses import dataclass

@dataclass
class PredictionRecord:
    prediction: float
    target: float

    @property
    def error(self) -> float:
        return 0.0  # TODO

record = PredictionRecord(3.5, 2.0)
assert record.error == 1.5
print("passed")

#### 10. Chapter Project
**Goal:** Build small experiment-card records using type hints, comprehensions, and dataclasses.

**Requirements:** define a dataclass for a trial result; store name, learning rate, and loss values; add a method returning the final loss; build a list of trial records and select the best one.

**Stretch Goals:** add `field(default_factory=list)` for optional histories; add a computed property for improvement from first to last loss.

**Evaluation Checklist:** equality compares fields; histories are not shared; best-trial selection is verified by assert; type hints identify every field.

In [ ]:
from dataclasses import dataclass

@dataclass
class TrialResult:
    name: str
    lr: float
    losses: list[float]

    def final_loss(self) -> float:
        return 0.0  # TODO

trials = [TrialResult("a", 0.1, [1.0, 0.4]), TrialResult("b", 0.01, [1.0, 0.3])]
best = None  # TODO: pick the trial with the smallest final loss.
assert best.name == "b"
print("project check passed")

#### 11. Solutions Appendix

--- SOLUTIONS: DO NOT PEEK UNTIL ATTEMPTED ---

In [ ]:
# Standalone Exercises 1-3
from dataclasses import dataclass, field

@dataclass
class User:
    name: str
    active: bool

assert User("Ada", True) == User("Ada", True)

@dataclass
class BatchOptions:
    size: int
    shuffle: bool = True

assert BatchOptions(16).shuffle is True

@dataclass
class TrialHistory:
    values: list[float] = field(default_factory=list)

first = TrialHistory()
second = TrialHistory()
first.values.append(0.1)
assert second.values == []
print("solutions passed")

In [ ]:
# Standalone Exercises 4-5 and Applied Drill
from dataclasses import dataclass

@dataclass
class ErrorPair:
    prediction: float
    target: float

    def absolute_error(self) -> float:
        return abs(self.prediction - self.target)

assert ErrorPair(2.0, 2.25).absolute_error() == 0.25

@dataclass(frozen=True)
class Token:
    text: str
    index: int

assert Token("hi", 0) == Token("hi", 0)
assert Token.__dataclass_params__.frozen is True

@dataclass
class TrainingConfig:
    examples: int
    batch_size: int
    lr: float = 0.1

    def steps_per_epoch(self) -> int:
        return self.examples // self.batch_size

assert TrainingConfig(20, 5).steps_per_epoch() == 4
print("solutions passed")

In [ ]:
# Compounding Drill and Chapter Project approach
from dataclasses import dataclass

@dataclass
class PredictionRecord:
    prediction: float
    target: float

    @property
    def error(self) -> float:
        return abs(self.prediction - self.target)

assert PredictionRecord(3.5, 2.0).error == 1.5

@dataclass
class TrialResult:
    name: str
    lr: float
    losses: list[float]

    def final_loss(self) -> float:
        return self.losses[-1]

trials = [TrialResult("a", 0.1, [1.0, 0.4]), TrialResult("b", 0.01, [1.0, 0.3])]
best = min(trials, key=lambda trial: trial.final_loss())
assert best.name == "b"
print("project solution passed")